# PDF renamer

Run all cells, review the table at the bottom, then press **Rename**. Nothing changes on disk
until you do. Everything runs locally, and OCR only runs when a PDF has no usable text or title.

In [ ]:
from pathlib import Path

FOLDER = Path("./test-data")
#Path.home() / "Downloads"
RECURSIVE = False
NAME_TEMPLATE = "{date} {title}.pdf"
MAX_TITLE_LEN = 80
DAY_FIRST = False  # how to read ambiguous dates like 03/04/2024
TEXT_PAGES = 3  # pages read for date, keywords and summary
OCR_ENABLED = True
OCR_DPI = 200

In [ ]:
import csv
import re
from collections import Counter
from datetime import date, datetime
from functools import lru_cache
from statistics import median

import anywidget
import numpy as np
import pandas as pd
import pymupdf
import traitlets
from dateutil import parser as dateparser
from rapidocr import RapidOCR

pymupdf.TOOLS.mupdf_display_errors(False)

## Reading
Pages become `(text, size, y)` lines, from the PDF text or from OCR, so later steps treat both the same.

In [ ]:
ocr_engine = lru_cache(maxsize=1)(lambda: RapidOCR(params={"Global.log_level": "error"}))


def page_lines(page):
    lines = []
    for block in page.get_text("dict")["blocks"]:
        for line in block.get("lines", []):
            if text := " ".join("".join(s["text"] for s in line["spans"]).split()):
                lines.append((text, max(s["size"] for s in line["spans"]), line["bbox"][1] / page.rect.height))
    return lines


def ocr_lines(page):
    pix = page.get_pixmap(dpi=OCR_DPI, colorspace=pymupdf.csRGB, alpha=False)
    image = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)
    result = ocr_engine()(image[:, :, ::-1])  # RapidOCR expects BGR
    if result.txts is None:
        return []
    return [(text, float(box[3][1] - box[0][1]), float(box[0][1]) / pix.height) for box, text in zip(result.boxes, result.txts)]


def to_text(lines):
    return "\n".join(line[0] for line in lines)

## Title
Metadata if its words are on page 1, then the largest text near the top, then OCR, then the file name.

In [ ]:
JUNK_TITLE = re.compile(
    r"^\s*(untitled|document\d*|powerpoint presentation|slide\s*\d*|page\s*\d*|scan\w*|image\d*|title|default|none|abstract|contents)\s*$"
    r"|^\s*microsoft\s+\w+|\.\w{2,4}\s*$|^\s*/|^[a-z]:\\|^[0-9a-f-]{16,}$|cid:|�|arxiv:|^\s*doi\b|https?:|www\.|©|copyright",
    re.IGNORECASE,
)


def usable(text):
    letters = sum(c.isalpha() for c in text)
    return letters >= 4 and letters / len(text) >= 0.5 and not JUNK_TITLE.search(text)


def largest_text(lines):
    """Biggest text in the top 60% of the page, and whether it stands out from the body text."""
    top = [line for line in lines if line[2] < 0.6 and usable(line[0])]
    if not top:
        return None, False
    size = max(line[1] for line in top)
    start = next(i for i, line in enumerate(top) if line[1] >= size * 0.95)
    parts = [top[start]]
    for line in top[start + 1 :]:
        if line[1] < size * 0.95 or line[2] - parts[-1][2] > 0.08:
            break
        parts.append(line)
    return " ".join(p[0] for p in parts), size > median(line[1] for line in lines) * 1.1


def on_page(title, text):
    words = re.findall(r"\w{3,}", title.lower())
    return sum(w in text.lower() for w in words) >= len(words) / 2 > 0


def pick_title(meta_title, lines, path, ocr):
    text, stands_out = largest_text(lines)
    if usable(meta_title) and on_page(meta_title, to_text(lines)):
        return meta_title, "metadata"
    if stands_out:
        return text, "ocr" if ocr else "page"
    if usable(meta_title):
        return meta_title, "metadata, not on page"
    if text:
        return text, "first line"
    return DATE.sub(" ", re.sub(r"[_+]+|%20|\s*\(\d+\)$", " ", path.stem)), "file name"


def clean_title(text):
    text = " ".join(re.sub(r'[<>:"/\\|?*\x00-\x1f]', " ", text).split()).strip(" .-_,;")
    text = text.title() if text.isupper() else text
    return text if len(text) <= MAX_TITLE_LEN else text[:MAX_TITLE_LEN].rsplit(" ", 1)[0].rstrip(" .-_,;")

## Date
Dates in the text, labelled ones first, then the file name, the PDF creation date and the file's
modified time. Birth, due and expiry dates are ignored.

In [ ]:
MONTH = r"(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*"
YEAR = r"(?:19|20)\d\d"
DATE = re.compile(
    rf"(?<![\d/.-])(?:{YEAR}[-_./]\d{{1,2}}[-_./]\d{{1,2}}|{YEAR}\d{{4}}|\d{{1,2}}[/.-]\d{{1,2}}[/.-]{YEAR}"
    rf"|\d{{1,2}}(?:st|nd|rd|th)?[\s-]+(?:of\s+)?{MONTH}\.?,?[\s-]+{YEAR}"
    rf"|(?<![a-z]){MONTH}\.?\s+\d{{1,2}}(?:st|nd|rd|th)?,?\s+{YEAR}|(?<![a-z]){MONTH}\.?,?\s+{YEAR})(?!\d)",
    re.IGNORECASE,
)
DATE_IGNORE = re.compile(r"birth|dob\b|expir|due|valid|until|renew|deadline|born", re.IGNORECASE)
DATE_LABEL = re.compile(r"date|published|issued|effective|statement|submitted|revised|as of", re.IGNORECASE)


def find_date(text):
    hits = []
    for match in DATE.finditer(text):
        found, before = match.group().replace("_", "-"), text[max(0, match.start() - 25) : match.start()]
        if DATE_IGNORE.search(before):
            continue
        try:  # ISO dates start with the year, and dayfirst would swap their month and day
            value = dateparser.parse(found, dayfirst=DAY_FIRST and not found[:4].isdigit(), default=datetime(2000, 1, 1)).date()
        except (ValueError, OverflowError):
            continue
        if date(1980, 1, 1) <= value <= date.today():
            has_day = len(re.findall(r"\d+", found)) > 1 or found.isdigit()
            hits.append((bool(DATE_LABEL.search(before)), has_day, value.strftime("%Y-%m-%d" if has_day else "%Y-%m")))
    return max(hits, key=lambda hit: hit[:2])[2] if hits else None


def pick_date(text, path, meta):
    if found := find_date(text):
        return found, "text"
    if found := find_date(path.stem):
        return found, "file name"
    if match := re.match(r"D:(\d{4})(\d\d)(\d\d)", meta.get("creationDate") or ""):
        return "-".join(match.groups()), "metadata"
    return datetime.fromtimestamp(path.stat().st_mtime).strftime("%Y-%m-%d"), "file modified"

## Summary
The sentences with the most frequent words, in document order. Stop words are English only.

In [ ]:
STOPWORDS = set(
    """about above after again also and any are because been before being below between both but can could did does
    down during each few for from further had has have having her here him his how into its just may might more most
    must not now off once only other our out over own same she should some such than that the their them then there
    these they this those through too under until very was were what when where which while who why will with would
    you your figure page using used""".split()
)


def summarize(text, count=3):
    flat = " ".join(text.split())
    freq = Counter(w for w in re.findall(r"[^\W\d_]{3,}", flat.lower()) if w not in STOPWORDS)
    sentences = list(dict.fromkeys(s for s in re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", flat) if 40 <= len(s) <= 400))

    def score(sentence):
        words = re.findall(r"[^\W\d_]{3,}", sentence.lower())
        return sum(freq[w] for w in words) / (len(words) + 5)

    top = sorted(sentences, key=score, reverse=True)[:count]
    return " ".join(s for s in sentences if s in top), ", ".join(w for w, _ in freq.most_common(6))

## Scan
Results are saved to `pdf-renamer-scan_<timestamp>.csv` in the scanned folder.

In [ ]:
def scan_pdf(path):
    with pymupdf.open(path) as doc:
        if doc.needs_pass:
            raise ValueError("password protected")
        meta, pages = doc.metadata or {}, list(doc)[:TEXT_PAGES]
        all_lines = [page_lines(page) for page in pages]
        first = all_lines[0] if all_lines else []
        ocr = OCR_ENABLED and bool(pages) and (len(to_text(first)) < 30 or "�" in to_text(first) or "(cid:" in to_text(first))
        if ocr:  # no text layer, or an unreadable one
            all_lines = [ocr_lines(page) for page in pages[:2]]
            first = all_lines[0]
        title, title_source = pick_title((meta.get("title") or "").strip(), first, path, ocr)
        if OCR_ENABLED and pages and not ocr and title_source in ("first line", "file name"):  # the title may be an image
            ocr_title, stands_out = largest_text(ocr_lines(pages[0]))
            if stands_out:
                title, title_source, ocr = ocr_title, "ocr", True
        text ="\n".join(to_text(lines) for lines in all_lines)
        date_value, date_source = pick_date(text, path, meta)
        summary, keywords = summarize(text)
        title = clean_title(title) or clean_title(path.stem)
        return {
            "suggested": NAME_TEMPLATE.format(date=date_value, title=title),
            "confidence": "high" if title_source in ("metadata", "page") and date_source != "file modified" else "low",
            "title": title,
            "title_source": title_source,
            "date": date_value,
            "date_source": date_source,
            "author": (meta.get("author") or "").strip(),
            "pages": doc.page_count,
            "ocr": ocr,
            "keywords": keywords,
            "summary": summary,
        }


def scan_row(path):
    row = {"path": path, "file": path.name, "error": None}
    try:
        return row | scan_pdf(path)
    except Exception as exc:
        return row | {"error": str(exc) if isinstance(exc, ValueError) else f"could not read: {type(exc).__name__}"}


def make_unique(rows):
    taken = set()
    for row in (r for r in rows if not r["error"]):
        name, n = row["suggested"], 2
        while (row["path"].parent, name.lower()) in taken or (name.lower() != row["file"].lower() and row["path"].with_name(name).exists()):
            name, n = f"{row['suggested'][:-4]} ({n}).pdf", n + 1
        taken.add((row["path"].parent, name.lower()))
        row["suggested"] = name
    return rows


paths = sorted(p for p in FOLDER.glob("**/*.pdf" if RECURSIVE else "*.pdf", case_sensitive=False) if p.is_file())
rows = make_unique([scan_row(p) for p in paths])
scanned = pd.DataFrame(rows)

# Each scan gets its own timestamp, shared by its saved results and its rename log
RUN_STAMP = datetime.now().strftime("%Y-%m-%d_%H%M%S")
SCAN_CSV = FOLDER / f"pdf-renamer-scan_{RUN_STAMP}.csv"
RENAME_LOG = FOLDER / f"pdf-renamer-log_{RUN_STAMP}.csv"
scanned.to_csv(SCAN_CSV, index=False, encoding="utf-8-sig")
print(f"Scanned {len(rows)} PDFs. Saved results to {SCAN_CSV}")

## Rename and undo
Existing files are never overwritten, and every rename is logged to `pdf-renamer-log_<timestamp>.csv`.
`undo_renames()` reverses the newest log, or one you pass in, and keeps it with an `undone_` prefix.

In [ ]:
def rename_file(source, name):
    """Returns the new path, or an error message if the file was left alone."""
    name = name.strip()
    name += "" if name.lower().endswith(".pdf") else ".pdf"
    if re.search(r'[<>:"/\\|?*\x00-\x1f]', name) or len(name) < 5:
        return "invalid file name"
    target = source.with_name(name)
    if target.exists():
        return "name already exists"
    try:
        source.rename(target)
    except OSError as exc:
        return exc.strerror or str(exc)
    new_log = not RENAME_LOG.exists()
    with RENAME_LOG.open("a", newline="", encoding="utf-8") as handle:
        log = csv.writer(handle)
        if new_log:
            log.writerow(["time", "old_path", "new_path"])
        log.writerow([datetime.now().isoformat(timespec="seconds"), source.resolve(), target.resolve()])
    return target


def undo_renames(log=None):
    log = log or max(FOLDER.glob("pdf-renamer-log_*.csv"), default=None)
    if log is None:
        return print("No rename log found.")
    with log.open(newline="", encoding="utf-8") as handle:
        for entry in reversed(list(csv.DictReader(handle))):
            old, new = Path(entry["old_path"]), Path(entry["new_path"])
            if new.exists() and not old.exists():
                new.rename(old)
                print(f"{new.name} -> {old.name}")
            else:
                print(f"SKIPPED {new.name}")
    log.rename(log.with_name(f"undone_{log.name}"))

## Review
High-confidence rows start ticked. Click a row for its summary, and edit any name in place.

In [ ]:
class RenameTable(anywidget.AnyWidget):
    rows = traitlets.List().tag(sync=True)
    _esm = r"""
    const esc = s => String(s ?? "").replace(/[&<>"']/g, c => `&#${c.charCodeAt(0)};`);
    export default { render({ model, el }) {
      let rows = [], query = "", filter = "All", open = null;
      el.innerHTML = `<div class="pr"><header><b>PDF renamer</b><span class="sub"></span>
        <input class="search" type="search" placeholder="Search"><nav></nav><button class="go"></button></header>
        <div class="wrap"><table><thead><tr><th></th><th>New name</th><th>Current file</th><th>Date</th><th>Confidence</th></tr></thead>
        <tbody></tbody></table></div></div>`;
      const $ = s => el.querySelector(s), body = $("tbody"), go = $(".go");
      const kind = r => r.error ? "Problems" : r.status === "renamed" ? "Renamed" : r.confidence === "high" ? "High" : "Low";
      const open_rows = () => rows.filter(r => !r.error && r.status !== "renamed");
      const count = () => { const n = open_rows().filter(r => r.on).length;
        go.textContent = n ? `Rename ${n}` : "Rename"; go.disabled = !n; };
      const detail = r => `<tr class="detail"><td></td><td colspan="4"><p>${esc(r.summary) || "No summary available."}</p><dl>
        ${[["Title", r.title, r.title_source], ["Date", r.date, r.date_source], ["Author", r.author], ["Pages", r.pages], ["Keywords", r.keywords]]
          .filter(f => f[1]).map(f => `<dt>${f[0]}</dt><dd>${esc(f[1])}${f[2] ? ` <i>from ${esc(f[2])}</i>` : ""}</dd>`).join("")}</dl></td></tr>`;
      const row = r => { const k = kind(r), busy = r.error || r.status === "renamed";
        return `<tr data-id="${r.id}" class="${busy ? "off" : ""}"><td><input type="checkbox" ${r.on ? "checked" : ""} ${busy ? "disabled" : ""}></td>
          <td>${busy ? esc(r.error || r.file) : `<input class="name" value="${esc(r.name)}" aria-label="New name">`}
          ${r.status && !busy ? `<div class="err">${esc(r.status)}</div>` : ""}</td>
          <td class="file" title="${esc(r.file)}">${esc(r.old_file ?? r.file)}</td><td class="date">${esc(r.date)}</td>
          <td><span class="pill ${k}">${k}${r.ocr ? " · OCR" : ""}</span></td></tr>${open === r.id ? detail(r) : ""}`; };
      const draw = () => {
        const q = query.toLowerCase();
        const shown = rows.filter(r => (filter === "All" || kind(r) === filter) &&
          [r.name, r.file, r.title, r.keywords, r.summary].join(" ").toLowerCase().includes(q));
        body.innerHTML = shown.map(row).join("") || `<tr><td colspan="5" class="none">No matching PDFs</td></tr>`;
        $("nav").innerHTML = ["All", "High", "Low", "Problems", "Renamed"].map(f => {
          const n = f === "All" ? rows.length : rows.filter(r => kind(r) === f).length;
          return f === "All" || n ? `<button data-f="${f}" class="${f === filter ? "on" : ""}">${f} ${n}</button>` : ""; }).join("");
        $(".sub").textContent = `${rows.length} files`;
        count(); };
      const sync = () => { const old = new Map(rows.map(r => [r.id, r]));
        rows = model.get("rows").map(r => ({ ...r, name: old.get(r.id)?.name ?? r.suggested,
          old_file: old.get(r.id)?.old_file ?? r.file,
          on: r.status === "renamed" || r.error ? false : old.get(r.id)?.on ?? r.confidence === "high" }));
        draw(); };
      const find = e => rows.find(r => r.id === +e.target.closest("[data-id]")?.dataset.id);
      el.addEventListener("input", e => { if (e.target.matches(".search")) { query = e.target.value; draw(); }
        else if (e.target.matches(".name")) find(e).name = e.target.value; });
      el.addEventListener("change", e => { if (e.target.type === "checkbox") { find(e).on = e.target.checked; count(); } });
      el.addEventListener("click", e => {
        if (e.target.dataset.f) { filter = e.target.dataset.f; draw(); }
        else if (e.target === go) { go.disabled = true;
          model.send({ rename: open_rows().filter(r => r.on).map(r => ({ id: r.id, name: r.name })) }); }
        else if (e.target.closest("tbody tr[data-id]") && !e.target.matches("input")) { const id = find(e).id;
          open = open === id ? null : id; draw(); } });
      model.on("change:rows", sync); sync();
    } };
    """
    _css = """
    .pr { --bg:#fff; --fg:#17181c; --mute:#6b7080; --line:#e7e8ee; --soft:#f5f6fa; --accent:#4f46e5; --on:#fff;
      --hi:#dcf5e7; --hit:#116a3c; --lo:#fdf0d5; --lot:#8a5a00; --bad:#fde4e4; --badt:#a61b1b;
      color-scheme:light; font:13px/1.45 system-ui,"Segoe UI",sans-serif; color:var(--fg); background:var(--bg);
      border:1px solid var(--line); border-radius:14px; overflow:hidden; }
    @media (prefers-color-scheme:dark) { .pr { --bg:#16171c; --fg:#eceef3; --mute:#9096a6; --line:#2b2d36; --soft:#1f2128;
      --accent:#818cf8; --on:#12131a; --hi:#123524; --hit:#6ee7a8; --lo:#3b2d0c; --lot:#f5c963; --bad:#401a1a; --badt:#fca5a5;
      color-scheme:dark; } }
    .pr header { display:flex; flex-wrap:wrap; align-items:center; gap:10px; padding:14px 16px; border-bottom:1px solid var(--line); }
    .pr header b { font-size:15px; } .pr .sub { color:var(--mute); margin-right:auto; }
    .pr input.search, .pr input.name { font:inherit; color:inherit; background:var(--soft); border:1px solid transparent; border-radius:8px; padding:6px 10px; outline:none; box-sizing:border-box; }
    .pr input.search { width:170px; } .pr input::placeholder { color:var(--mute); } .pr input.name { width:100%; min-width:260px; }
    .pr input.search:focus, .pr input.name:focus { border-color:var(--accent); background:var(--bg); }
    .pr nav { display:flex; gap:2px; background:var(--soft); border-radius:9px; padding:3px; }
    .pr nav button { font:inherit; color:var(--mute); background:none; border:0; border-radius:7px; padding:4px 9px; cursor:pointer; }
    .pr nav button.on { color:var(--fg); background:var(--bg); box-shadow:0 1px 2px #0002; }
    .pr .go { font:inherit; font-weight:600; color:var(--on); background:var(--accent); border:0; border-radius:8px; padding:7px 16px; cursor:pointer; }
    .pr .go:disabled { opacity:.4; cursor:default; }
    .pr .wrap { max-height:70vh; overflow:auto; }
    .pr table { width:100%; border-collapse:collapse; }
    .pr th { position:sticky; top:0; z-index:1; background:var(--bg); color:var(--mute); font-weight:500; text-align:left; padding:8px 10px; border-bottom:1px solid var(--line); }
    .pr td { padding:7px 10px; border-bottom:1px solid var(--line); vertical-align:middle; background:none; }
    .pr tbody tr[data-id] { cursor:pointer; } .pr tbody tr[data-id]:hover { background:var(--soft); }
    .pr tr.off td { color:var(--mute); }
    .pr td.file { max-width:240px; overflow:hidden; white-space:nowrap; text-overflow:ellipsis; color:var(--mute); }
    .pr td.date { white-space:nowrap; font-variant-numeric:tabular-nums; }
    .pr .pill { display:inline-block; white-space:nowrap; border-radius:99px; padding:1px 9px; font-size:12px; font-weight:500; }
    .pr .High, .pr .Renamed { background:var(--hi); color:var(--hit); } .pr .Low { background:var(--lo); color:var(--lot); }
    .pr .Problems { background:var(--bad); color:var(--badt); } .pr .err { color:var(--badt); font-size:12px; margin-top:3px; }
    .pr tr.detail td { background:var(--soft); cursor:default; } .pr tr.detail p { margin:2px 0 8px; max-width:80ch; }
    .pr dl { display:grid; grid-template-columns:max-content 1fr; gap:2px 12px; margin:0; } .pr dt { color:var(--mute); } .pr dd { margin:0; }
    .pr dd i { color:var(--mute); } .pr .none { text-align:center; color:var(--mute); padding:28px; }
    """

    def __init__(self, rows):
        super().__init__()
        self.source = rows
        self.refresh()
        self.on_msg(self.handle)

    def refresh(self):
        self.rows = [{k: v for k, v in row.items() if k != "path"} | {"id": i} for i, row in enumerate(self.source)]

    def handle(self, _, content, __):
        for item in content.get("rename", []):
            row = self.source[item["id"]]
            result = rename_file(row["path"], item["name"])
            if isinstance(result, Path):
                row |= {"path": result, "file": result.name, "status": "renamed"}
            else:
                row["status"] = result
        self.refresh()


table = RenameTable(rows)
table